In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
for i in range(5):
    print(files[i].filename)

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/04-dataset.md
01-agentic-rag/lessons/05-search.md


In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
len(documents)

72

In [7]:
import sys
sys.path.append("..")

from rag_pipeline import RAGBase
rag_doc = RAGBase(documents)

In [8]:
index = rag_doc.build_index()
result = index.search("How does the agentic loop keep calling the model until it stops?" , num_results=3)
for r in range(3):
    print(result[r].get("filename"))


01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md


In [9]:
response , answer = rag_doc.rag("How does the agentic loop keep calling the model until it stops?")

In [10]:
print(response.usage_metadata.prompt_token_count)

7944


In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [12]:
rag_chunks = RAGBase(chunks)
index = rag_chunks.build_index()
index.search("How does the agentic loop keep calling the model until it stops?" , num_results=3)
for r in range(3):
    print(result[r].get("filename"))

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md


In [13]:
response , answer = rag_chunks.rag("How does the agentic loop keep calling the model until it stops?")

In [14]:
print(response.usage_metadata.prompt_token_count)

2597


In [15]:
from openai import OpenAI

from toyaikit.llm import OpenAIChatCompletionsClient 
from toyaikit.chat.runners import OpenAIChatCompletionsRunner
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [16]:
agent_tools = Tools()
agent_tools.add_tool(rag_chunks.search)

In [17]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'},
    'num_results': {'type': 'string', 'description': 'num_results parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [22]:
from dotenv import load_dotenv
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [23]:
#we use the openAi format and syntax , but the backend is calling the gemini api, since the toyaikit library is built for openai api
gemini_client = OpenAI(
    api_key= GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [24]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIChatCompletionsClient(
        model="gemini-2.5-flash-lite",
        client=gemini_client,
    )
)

In [27]:
resultat = runner.loop(
    prompt= "How does the agentic loop work, and how is it different from plain RAG?",
    callback = callback
)

-> Response received


-> Response received
